In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    mean_squared_error,
    r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor

In [3]:
df = pd.read_csv('rawdata.csv')

print(df.head())

print(df.info())

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOptionLevel  \
0  ...

## Data Cleaning

In [4]:
#Turned specific columns into numerical variable 
df['Attrition'] = df['Attrition'].replace({'Yes': 1, 'No': 0})
df['OverTime'] = df['OverTime'].replace({'Yes': 1, 'No': 0})
df['Gender'] = df['Gender'].replace({'Male': 1, 'Female': 0})
print(df[['Attrition', 'OverTime', 'Gender']].head())

  Attrition OverTime Gender
0         1        1      0
1         0        0      1
2         1        1      1
3         0        1      0
4         0        0      1


In [5]:
#removing unecessary data
drop_cols = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df = df.drop(columns=drop_cols)

In [6]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

Missing values: 0
Duplicates: 0


In [7]:
print(df.head())
print(df.info())

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41         1      Travel_Rarely       1102                   Sales   
1   49         0  Travel_Frequently        279  Research & Development   
2   37         1      Travel_Rarely       1373  Research & Development   
3   33         0  Travel_Frequently       1392  Research & Development   
4   27         0      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EnvironmentSatisfaction Gender  \
0                 1          2  Life Sciences                        2      0   
1                 8          1  Life Sciences                        3      1   
2                 2          2          Other                        4      1   
3                 3          4  Life Sciences                        4      0   
4                 2          1        Medical                        1      1   

   ...  PerformanceRating  RelationshipSatisfaction  StockOptionLeve

In [8]:
# checking Class Imbalance
attrition_counts = df['Attrition'].value_counts()
attrition_pcts = df['Attrition'].value_counts(normalize=True) * 100

imbalance_summary = pd.DataFrame({
    'Count': attrition_counts,
    'Percentage (%)': attrition_pcts.round(2)
})
print("\nAttrition:")
print(imbalance_summary)


Attrition:
           Count  Percentage (%)
Attrition                       
0           1233           83.88
1            237           16.12


In [9]:
print(" Attrition ανά Τμήμα ")
print((df.groupby('Department')['Attrition'].mean() * 100).sort_values(ascending=False).round(2))

print("\n Attrition ανά Ρόλο Εργασίας ")
print((df.groupby('JobRole')['Attrition'].mean() * 100).sort_values(ascending=False).round(2))

 Attrition ανά Τμήμα 
Department
Sales                     20.63
Human Resources           19.05
Research & Development    13.84
Name: Attrition, dtype: object

 Attrition ανά Ρόλο Εργασίας 
JobRole
Sales Representative         39.76
Laboratory Technician        23.94
Human Resources              23.08
Sales Executive              17.48
Research Scientist            16.1
Manufacturing Director         6.9
Healthcare Representative     6.87
Manager                        4.9
Research Director              2.5
Name: Attrition, dtype: object


In [11]:
focus_cats = ['Department', 'JobRole', 'JobLevel', 'EnvironmentSatisfaction', 'PerformanceRating']
for i in focus_cats:
    print(f"\n Attrition ανά {i} ")
    summary = df.groupby(i)['Attrition'].agg(
        Headcount='count', #πλήθος γραμμών
        Attrition_Count='sum', #πόσοι έφυγαν
        Attrition_Rate=lambda x: (x.mean() * 100).round(2) #ποσοστό αποχώρησης
    ).reset_index()
    print(summary.sort_values(by='Attrition_Rate', ascending=False).to_string(index=False))


 Attrition ανά Department 
            Department  Headcount Attrition_Count  Attrition_Rate
                 Sales        446              92           20.63
       Human Resources         63              12           19.05
Research & Development        961             133           13.84

 Attrition ανά JobRole 
                  JobRole  Headcount Attrition_Count  Attrition_Rate
     Sales Representative         83              33           39.76
    Laboratory Technician        259              62           23.94
          Human Resources         52              12           23.08
          Sales Executive        326              57           17.48
       Research Scientist        292              47           16.10
   Manufacturing Director        145              10            6.90
Healthcare Representative        131               9            6.87
                  Manager        102               5            4.90
        Research Director         80               2          